# **Atividade Avaliativa** 
## **Instituto de Computação** 
## **Universidade Estadual de Campinas \- UNICAMP** 
### **Curso: INF0087** Sistemas Inteligentes com Agentes Autônomos usando Grandes Modelos de Linguagem
### **Disciplina: Fundamentos de IA Generativa e PLN**

<br>

#### **1. Descrição da Atividade:**

* Realizar o ajuste fino:
  * Usar um LLM pequeno como base, ex.: TinyLlama/TinyLlama-1.1B-Chat-v1.0, google/gemma-3-1b-pt …  
  * Para fazer o fine-tuning supervisionado, utilizar o dataset “HuggingFaceTB/smoltalk2”, subconjunto/subset “SFT” e Split “sft\_smoltalk\_multilingual\_8languages\_lang\_5\_no\_think” (link: [https://huggingface.co/datasets/HuggingFaceTB/smoltalk2/viewer/SFT/smoltalk\_multilingual\_8languages\_lang\_5\_no\_think](https://huggingface.co/datasets/HuggingFaceTB/smoltalk2/viewer/SFT/smoltalk_multilingual_8languages_lang_5_no_think)). Dica: baixe o dataset no seu gdrive (ou local) para não precisar baixá-lo a cada execução.   
  * Aplicar QLoRA para realizar o ajuste fino de instruções.  
  * Ilustrar, com exemplos, as diferenças de geração entre o modelo de base e o ajustado.

<br>

#### **2. Critérios de Avaliação:** 
Os seguintes critérios serão considerados com igual peso (25% cada):

* Complexidade:   
  * Por ser uma atividade aberta, pode ser executada em diferentes níveis de complexidade. Por exemplo: um ajuste mais refinado dos parâmetros do QLoRA, uma análise comparativa dos modelos ou a filtragem dos dados de treinamento acrescentam nota, mas não são essenciais.     
* Completude:  
  * A atividade completa inclui carregar a LLM de base, carregar o conjunto de dados para o SFT, realizar o ajuste fino com QLoRA e verificar as diferenças nas respostas. O ajuste por preferências não faz parte da avaliação (não vale nota) e é uma atividade extra para os alunos que desejarem testar suas habilidades.    
* Corretude:  
  * Ausência de bugs, código eficiente e uso adequado da linguagem Python e bibliotecas.   
* Documentação:  
  * Os códigos deverão ser acompanhados de documentação que explique cada passo no notebook, ou seja, intercalando blocos de texto e de código. A documentação mais completa e clara receberá uma nota maior.

Importante: não fazem parte da avaliação aspectos que dependem apenas do poder computacional disponível (ex.: um conjunto de treinamento muito grande não influencia a nota).

<br>

#### **3. Modo de Entrega:**

**A atividade é Individual.**
Os alunos deverão copiar este notebook, incluir documentação (blocos de texto) e códigos, abaixo deste enunciado e também, as saídas de cada bloco de código.    

A entrega do notebook (.ipynb) será feita por meio da atividade no Google ClassRoom.  


**Renato Bueno Domingos de Oliveira**

===============================================
# Passo 1: Instalar as bibliotecas necessárias.
===============================================
Instalamos as bibliotecas principais para quantização, fine-tuning eficiente e manipulação de datasets.

In [ ]:
!pip install -q -U bitsandbytes peft trl accelerate datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.4 MB/s eta 0:00:00


# Passo 2: Autenticação no Hugging Face
Para usar a família de modelos Gemma, precisamos autenticar nosso ambiente usando um Token do Hugging Face. Aceite dos termos de uso do modelo google/gemma-3-1b-pt e google/gemma-3-1b-it no portal do Hugging Face

In [ ]:
import os
from huggingface_hub import login

In [ ]:
# Substituindo pela minha chave de acesso (Token) do Hugging Face
HF_TOKEN = "veja na pasta  G:\Meu Drive\Doutorado\cursos\SistemasInteligentes_UNICAMP\token.txts"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

In [ ]:
from huggingface_hub import login
login()


--------------------------------------------------------------------------------
Passo 3
----------------------------------------------------------------------
Baixar amostra do dataset e salvar no Google Drive

In [ ]:
from datasets import load_dataset
from itertools import islice
import json
import os

output_path = "/content/drive/MyDrive/Doutorado/cursos/SistemasInteligentes_UNICAMP"
os.makedirs(output_path, exist_ok=True)

arquivo_saida = os.path.join(output_path, "smoltalk2_2500.jsonl")

print("Carregando dataset em streaming...")

dataset_stream = load_dataset(
    "HuggingFaceTB/smoltalk2",
    "SFT",
    split="smoltalk_multilingual_8languages_lang_5_no_think",
    streaming=True
)

subset = islice(dataset_stream, 2500)

with open(arquivo_saida, "w", encoding="utf-8") as f:
    for exemplo in subset:
        f.write(json.dumps(exemplo, ensure_ascii=False) + "\n")

print("Dataset salvo em:", arquivo_saida)

Carregando dataset em streaming...
Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]
Dataset salvo em: /content/drive/MyDrive/Doutorado/cursos/SistemasInteligentes_UNICAMP/smoltalk2_2500.jsonl


4 - Recarregar o dataset salvo

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files=arquivo_saida,
    split="train"
)

print(dataset)
print(dataset[0]["messages"])

Generating train split: 0 examples [00:00, ? examples/s]
Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2500
})
[{'content': 'O diretor da escola, juntamente com os professores e demais funcionários, ficou muito satisfeito ao concluir com sucesso a difícil tarefa de organizar o festival escolar anual dentro do prazo, que foi muito apreciado pelos estudantes e seus responsáveis, que também estavam presentes.', 'role': 'user'}, {'content': 'O diretor da escola, em conjunto com os professores e demais funcionários, sentiu-se extremamente satisfeito ao concluir com êxito a árdua tarefa de organizar o festival escolar anual dentro do prazo estabelecido. O evento, que foi muito apreciado pelos estudantes e seus responsáveis, que também participaram ativamente, demonstrou o empenho e a dedicação de toda a equipe envolvida. A realização do festival não apenas proporcionou momentos de lazer e entretenimento, mas também fortaleceu o vínculo entre a comunida

Passo 5 —  Separar treino e teste

In [ ]:
dataset_splits = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset_splits)

DatasetDict({
    train: Dataset({
        features: ['messages', 'chat_template_kwargs', 'source'],
        num_rows: 2250
    })
    test: Dataset({
        features: ['messages', 'chat_template_kwargs', 'source'],
        num_rows: 250
    })
})



Passo 7 — Carregar modelo base quantizado em 4 bits


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "google/gemma-3-1b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, 
    bnb_4bit_use_double_quant=True,
)


tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

model.config.pad_token_id = tokenizer.pad_token_id
model.config.torch_dtype = torch.bfloat16

Loading weights: 100%


In [ ]:
print("torch_dtype config:", model.config.torch_dtype)
print("bnb compute dtype:", bnb_config.bnb_4bit_compute_dtype)

torch_dtype config: torch.bfloat16
bnb compute dtype: torch.bfloat16


Passo 8 — Configurar LoRA/QLoRA

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

Passo 9 — Função para testar o modelo antes do fine-tuning

In [ ]:
def gerar_resposta(model, tokenizer, pergunta, max_new_tokens=150):
    messages = [
        {"role": "user", "content": pergunta}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    resposta = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return resposta

Passo 10 — Testar o modelo base

In [ ]:
perguntas_teste = [
    "Explique o que é aprendizado de máquina para um aluno iniciante.",
    "Crie um exemplo simples de código Python para somar três números.",
    "Qual é a diferença entre aprendizado supervisionado e não supervisionado?"
]

respostas_base = {}

for pergunta in perguntas_teste:
    print("="*80)
    print("PERGUNTA:", pergunta)
    resposta = gerar_resposta(model, tokenizer, pergunta)
    respostas_base[pergunta] = resposta
    print("RESPOSTA DO MODELO BASE:")
    print(resposta)

PERGUNTA: Explique o que é aprendizado de máquina para um aluno iniciante.
RESPOSTA DO MODELO BASE:
user
Explique o que é aprendizado de máquina para um aluno iniciante.
model
## Aprendizado de Máquina para Iniciantes: Uma Explicação Simplificada

Imagine que você quer ensinar um computador a reconhecer fotos de gatos. Como faria isso? Você poderia simplesmente dizer "Isso é um gato!" e o computador faria a identificação, certo? Mas, a verdade é que a maioria dos computadores não consegue isso de forma muito boa.

**Aprendizado de Máquina (Machine Learning) é a chave para resolver isso.** Em vez de programar regras específicas para cada gato (como "gatos têm orelhas pontudas"), o aprendizado de máquina permite que o computador **aprenda** a identificar gatos analisando *muitos* exemplos de fotos de gatos.

**Em termos mais simples
PERGUNTA: Crie um exemplo simples de código Python para somar três números.
RESPOSTA DO MODELO BASE:
user
Crie um exemplo simples de código Python para somar

In [ ]:
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

dataset = dataset.map(format_chat)

dataset_splits = dataset.train_test_split(test_size=0.1, seed=42)

print(dataset_splits["train"][0]["text"])

Map: 100%
<bos><start_of_turn>user
Welche Geschichten können aus dem Thema Schatten entstehen?<end_of_turn>
<start_of_turn>model
Aus dem Thema Schatten können vielfältige und faszinierende Geschichten entstehen. Hier sind einige Beispiele:

1. **Der Schatten, der spricht**: In dieser Geschichte hat ein kleines Mädchen einen Schatten, der plötzlich anfängt, mit ihr zu sprechen. Der Schatten teilt Geheimnisse und Abenteuer mit ihr, die sie niemandem sonst erzählt. Zusammen erleben sie unglaubliche Momente, die das Mädchen auf eine magische Reise führen.

2. **Der verschwundene Schatten**: Ein Junge bemerkt eines Tages, dass sein Schatten verschwunden ist. Er begibt sich auf eine Suche, um ihn wiederzufinden, und stößt dabei auf eine geheime Welt, in der Schatten leben und ihre eigenen Geschichten haben. Er lernt, dass jeder Schatten eine besondere Bedeutung hat und dass sein eigenes Schattenwesen ihm wichtige Lektionen beibringen kann.

3. **Die Schattenstunde**: In einer alten Stadt gib

Passo 11 — Configurar o SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig
import os

os.environ["WANDB_DISABLED"] = "true"

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_splits["train"],

    args=SFTConfig(
        output_dir=os.path.join(output_path, "outputs-gemma-smoltalk"),

        dataset_text_field="text",

        max_length=256,
        packing=False,
        num_train_epochs=1,

        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        gradient_checkpointing=False,

        optim="paged_adamw_8bit",

        warmup_steps=25,
        learning_rate=1e-4,

        fp16=False, 
        bf16=False,  

        max_grad_norm=0.3,

        logging_steps=25,

        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        save_only_model=True,

        lr_scheduler_type="constant",

        report_to="none"
    ),

    peft_config=lora_config,
    processing_class=tokenizer,
)

In [ ]:
import torch

print(torch.cuda.get_device_name(0))
print("BF16 suportado?", torch.cuda.is_bf16_supported())

Tesla T4
BF16 suportado? True


Passo 12 — Treinar o modelo

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


TrainOutput(global_step=563, training_loss=1.3691976591283008, metrics={'train_runtime': 1646.1685, 'train_samples_per_second': 1.367, 'train_steps_per_second': 0.342, 'total_flos': 2316551323809024.0, 'train_loss': 1.3691976591283008})

Passo 13 — Salvar o adaptador LoRA no Google Drive

In [ ]:
modelo_saida = os.path.join(output_path, "gemma-smoltalk-qlora-adapter")

trainer.save_model(modelo_saida)
tokenizer.save_pretrained(modelo_saida)

print("Modelo ajustado salvo em:", modelo_saida)

Modelo ajustado salvo em: /content/drive/MyDrive/Doutorado/cursos/SistemasInteligentes_UNICAMP/gemma-smoltalk-qlora-adapter


Passo 14 — Testar o modelo ajustado

In [ ]:
respostas_ajustado = {}

for pergunta in perguntas_teste:
    print("="*80)
    print("PERGUNTA:", pergunta)
    resposta = gerar_resposta(model, tokenizer, pergunta)
    respostas_ajustado[pergunta] = resposta
    print("RESPOSTA DO MODELO AJUSTADO:")
    print(resposta)

PERGUNTA: Explique o que é aprendizado de máquina para um aluno iniciante.
RESPOSTA DO MODELO AJUSTADO:
user
Explique o que é aprendizado de máquina para um aluno iniciante.
model
O aprendizado de máquina (Machine Learning) é uma área da inteligência artificial que permite aos sistemas aprender a partir de dados sem serem explicitamente programados. Diferente dos métodos tradicionais de programação, onde as instruções são definidas de antemão, os algoritmos de aprendizado de máquina permitem que os sistemas aprendam e melhorem a partir de experiências.

Em termos mais simples, imagine que você quer ensinar um computador a reconhecer gatos em fotos. Você pode fornecer a ele um grande conjunto de fotos de gatos, mas sem essa informação, ele não saberá o que procurar. O aprendizado de máquina permite que o sistema aprenda a partir dessas fotos, identificando características únicas de gatos, como olhos grandes, orelhas
PERGUNTA: Crie um exemplo simples de código Python para somar três núme

Passo 15 — Comparar modelo base vs. modelo ajustado

In [ ]:
for pergunta in perguntas_teste:
    print("="*100)
    print("PERGUNTA:")
    print(pergunta)

    print("\n--- RESPOSTA DO MODELO BASE ---")
    print(respostas_base[pergunta])

    print("\n--- RESPOSTA DO MODELO AJUSTADO ---")
    print(respostas_ajustado[pergunta])

PERGUNTA:
Explique o que é aprendizado de máquina para um aluno iniciante.

--- RESPOSTA DO MODELO BASE ---
user
Explique o que é aprendizado de máquina para um aluno iniciante.
model
## Aprendizado de Máquina para Iniciantes: Uma Explicação Simplificada

Imagine que você quer ensinar um computador a reconhecer fotos de gatos. Como faria isso? Você poderia simplesmente dizer "Isso é um gato!" e o computador faria a identificação, certo? Mas, a verdade é que a maioria dos computadores não consegue isso de forma muito boa.

**Aprendizado de Máquina (Machine Learning) é a chave para resolver isso.** Em vez de programar regras específicas para cada gato (como "gatos têm orelhas pontudas"), o aprendizado de máquina permite que o computador **aprenda** a identificar gatos analisando *muitos* exemplos de fotos de gatos.

**Em termos mais simples

--- RESPOSTA DO MODELO AJUSTADO ---
user
Explique o que é aprendizado de máquina para um aluno iniciante.
model
O aprendizado de máquina (Machine Le

Passo 16 — Texto para documentação no notebook

## Análise dos resultados

Após o fine-tuning supervisionado com QLoRA, foi realizada uma comparação qualitativa entre o modelo de base e o modelo ajustado. As mesmas perguntas foram aplicadas aos dois modelos.

O objetivo da comparação foi observar se o modelo ajustado passou a gerar respostas mais alinhadas ao formato de instrução, com maior clareza, organização e adequação ao contexto conversacional. Como foi utilizada uma amostra reduzida de 2500 exemplos e apenas uma época de treinamento, não se espera uma transformação profunda do comportamento do modelo, mas sim indícios de adaptação ao padrão do dataset.

O uso de QLoRA permitiu realizar o ajuste fino com menor consumo de memória, pois o modelo base foi carregado em 4 bits e apenas pequenos adaptadores LoRA foram treinados. Essa estratégia é adequada para ambientes com recursos limitados, como o Google Colab.

**O que mudou entre o modelo base e o ajustado?**

*1. Respostas ficaram mais objetivas*

**Modelo base**
* Mais “conversacional”
* Usa metáforas
* Explica passo a passo
* Tenta ensinar com mais contexto

Exemplo:

“Imagine que você quer ensinar um computador a reconhecer fotos de gatos…”

**Modelo ajustado**

* Mais direto
* Linguagem mais técnica
* Menos narrativa
* Vai rapidamente ao conceito

Exemplo:

“O aprendizado de máquina é uma área da inteligência artificial…”

**O modelo ajustado parece mais “instruction tuned”**

O dataset SmolTalk foi criado justamente para ensinar:

* seguir instruções;
* responder perguntas;
* gerar respostas úteis;
* manter um estilo consistente.

O modelo ajustado:

* respondeu exatamente ao pedido;
* evitou excesso de texto;
* produziu respostas mais formatadas para QA/chat.

**O código Python melhorou**

Isso mostra:

* melhor aderência à instrução;
* maior simplicidade;
* resposta mais adequada ao nível solicitado.

**Conclusão:**
Os resultados obtidos demonstram que o ajuste fino supervisionado utilizando QLoRA foi capaz de modificar o comportamento do modelo base, tornando suas respostas mais objetivas, consistentes e alinhadas ao formato instrucional presente no dataset SmolTalk.